# 03. MAF でシングルエージェントを構築する (ツールなし)

## コードの解説

`Agent` クラスが MAF の中心的なコンポーネントです。
最小構成では `client`（LLM クライアント）、`name`（エージェント名）、`instructions`（システムプロンプト）を指定するだけでエージェントを作成できます。

### 主要なコンポーネント

| コンポーネント | クラス | 説明 |
|------------|------|------|
| エージェント | `Agent` | MAF の中心クラス。LLM クライアント・ツール・スキルを束ねる |
| LLM クライアント | `OpenAIChatCompletionClient` | Azure OpenAI に接続するクライアント |
| 実行 | `await agent.run(query)` | エージェントにクエリを送り、応答を得る |

### 認証方式の切り替え

環境変数 `USE_KEY_AUTH` で認証方式を切り替えられます：

| 値 | 認証方式 | 用途 |
|----|--------|------|
| `True` | API キー認証 | 開発・テスト環境向け |
| `False` | DefaultAzureCredential | 本番環境向け（Managed Identity など） |

### Agent の設定パラメータ

| パラメータ | 説明 |
|---------|------|
| `client` | LLM クライアント（OpenAI / Foundry など） |
| `name` | エージェントの識別名 |
| `instructions` | エージェントの役割・振る舞いを定義するシステムプロンプト |
| `tools` | エージェントが使えるツールのリスト（省略可） |

In [ ]:
import os
from dotenv import load_dotenv
from agent_framework import Agent
from agent_framework.openai import OpenAIChatCompletionClient

load_dotenv()

# 環境変数から接続情報を取得
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY  = os.getenv("AZURE_OPENAI_API_KEY")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

# 認証方式を環境変数で切り替える（True: API キー, False: DefaultAzureCredential）
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")
if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=AZURE_OPENAI_API_KEY)
    print("認証方式: API キー認証")
else:
    from azure.identity.aio import DefaultAzureCredential
    # フレームワークが AZURE_OPENAI_API_KEY を credential より優先するため明示的に削除
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    auth_kwargs = dict(credential=DefaultAzureCredential())
    print("認証方式: DefaultAzureCredential")


# ---------------------------------------------------------------
# MAF シングルエージェントの最小構成
#
# ポイント1: client に OpenAIChatCompletionClient を渡して Azure OpenAI に接続
# ポイント2: instructions にエージェントの役割を日本語で書く
# ポイント3: tools=[] は省略可（ツールなしの場合は LLM の知識だけで応答）
# ---------------------------------------------------------------
agent = Agent(
    client=OpenAIChatCompletionClient(
        **auth_kwargs,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        model=MODEL_DEPLOYMENT_NAME,
    ),
    name="TravelAssistant",
    instructions=(
        "あなたは親切な旅行アシスタントです。"
        "旅行先のおすすめや旅のコツをわかりやすく提供してください。"
    ),
    # tools=[] # ツールなしのためコメントアウト
)

# エージェントの実行（await で非同期実行）
response = await agent.run("東京旅行の見どころを 3 つ教えてください。")
print(response)